# Trabalho individual — Classificação da natureza de despesas eleitorais (TSE, 2024)

**Processamento de Dados Massivos — Prof. Alexandre Roriz | Spark ML | Classificação multiclasses**

**Problema:** prever a natureza oficial da despesa contratada (`DS_ORIGEM_DESPESA`) com atributos contextuais do lançamento e da candidatura, sem usar o próprio rótulo ou sua descrição textual como entrada.

**O que esta revisão corrige:**
1. Usa `TP_PRESTACAO_CONTAS` (coluna realmente encontrada) para reter **prestação Final**; registros parciais são excluídos antes da deduplicação.
2. Audita chaves repetidas e valores divergentes; não escolhe versões conflitantes arbitrariamente.
3. Calcula a cobertura das 5 classes com o denominador da **mesma amostra**.
4. Configura Java 17 no kernel do Windows, compatível com o PySpark 4.x.
5. Apresenta diagnóstico antecipado e salvamento por `save()` com mensagem clara quando faltam componentes Hadoop nativos no Windows.

> **Estado de validação:** código revisto e testado estaticamente. Sem execução do ZIP completo neste ambiente. As contagens/métricas serão **recalculadas**, e podem diferir das obtidas na versão anterior.


**Versão final para repositório GitHub:** mantém as 22 seções do trabalho, com correções
de limpeza, prevenção de vazamento, avaliação multiclasses e diagnóstico da gravação
Spark no Windows. Esta revisão foi validada estruturalmente e sintaticamente, **não
reexecutada com os 4,45 milhões de registros**. As saídas de execução não são
fabricadas: ao abrir o notebook, execute todas as células para reproduzi-las.


## 1. Contextualização e os 5 Vs (Unidades 1, 2 e 3)

**Fonte primária:** Tribunal Superior Eleitoral (TSE), Portal de Dados Abertos, conjunto **Prestação de Contas Eleitorais — 2024**, publicado pela ASEPA a partir do SPCE:

- [Página do conjunto de dados](https://dadosabertos.tse.jus.br/dataset/prestacao-de-contas-eleitorais-2024)
- [Download nacional oficial dos dados de candidatos (ZIP)](https://cdn.tse.jus.br/estatistica/sead/odsele/prestacao_contas/prestacao_de_contas_eleitorais_candidatos_2024.zip)
- Recurso utilizado dentro do ZIP: `despesas_contratadas_candidatos_2024_BRASIL.csv` (quando disponibilizado; há tratamento alternativo para arquivos por UF).

**Unidade de análise:** um registro de despesa contratada vinculada a uma candidatura; não confundir com pagamento efetuado. A quantidade real de registros e colunas será impressa abaixo, **depois de baixar o arquivo**.

**Por que o cenário pertence a dados massivos?**

| Dimensão | Exemplo neste projeto |
|---|---|
| Volume | Registros nacionais de despesas de centenas de milhares de campanhas; há muitas linhas e atributos. |
| Velocidade | Durante uma eleição, prestações de contas e retificações atualizam dados ao longo do tempo; aqui usamos uma fotografia estática (*batch*). |
| Variedade | Datas, valores com decimal brasileiro, categorias, UFs, fornecedores, candidaturas e textos. |
| Veracidade | Campos ausentes, revisões/retificações, valores inválidos e possíveis duplicidades precisam ser examinados. |
| Valor | Apoiar a compreensão de padrões de lançamento de gastos por categoria, com ressalvas metodológicas. |

**Arquitetura (Unidades 2 e 3):** Hadoop/HDFS e MapReduce ajudam a explicar o processamento paralelo; Spark organiza *partitions*, executa transformações lazy e otimiza trabalhos iterativos com Spark ML. **Neste Colab usamos `local[*]`: a API é distribuída, mas o teste roda em uma única máquina.** Não afirmamos ter usado um cluster multinó.

**Escopo:** análise de classificação de registros, sem qualquer pontuação de candidatos, campanhas ou partidos. O uso do Spark se justifica pela escalabilidade do cenário nacional e, principalmente, pelo processo repetitivo de preparação e treinamento que pode migrar para um cluster quando necessário.

**Unidade estatística:** lançamento de despesa contratada em prestação **Final**, após auditoria e consolidação de registros repetidos. Não estamos classificando a legalidade do gasto nem verificando se foi pago.

**Risco central:** o arquivo nacional pode conter prestações parciais e finais. Misturar versões ou escolher repetidos sem critério contamina a população do estudo. Esta revisão restringe o tipo e retira chaves com fatos conflitantes.


## 2. Ambiente — VS Code/Windows ou Google Colab (Unidade 4)

- No Windows, a sessão Jupyter pode herdar **Java 8** mesmo quando o CMD já aponta para Java 17. A célula seguinte identifica o JDK 17 antes de carregar o Spark.
- A máquina testada pelo aluno executou PySpark **4.2.0**, Java 17 e `local[2]`.
- No Colab/Linux, o Java deve estar instalado; se necessário, configure o ambiente Linux antes de executar.
- **Atenção:** ler e treinar no Windows pode funcionar enquanto `save()` falha sem utilitários nativos do Hadoop. A próxima célula verifica essa limitação com uma gravação pequena, antes de gastar tempo treinando.


**Publicação GitHub:** o notebook usa o ZIP oficial durante a execução, mas não o
inclui no repositório. O modelo é salvo em `modelos/` quando o ambiente suporta
escrita Hadoop. Coloque `modelos/`, `dados_tse_2024/` e `_teste_gravacao_spark_tse/`
no `.gitignore` do repositório. O campo de execução deve exibir `PERSISTENCIA_OK=True`
para comprovar o requisito de salvamento/recarregamento; se o Windows não
suportar a escrita, execute o notebook integralmente em Linux/Colab.


In [ ]:
import os
import sys
import glob
import shutil
import subprocess
import importlib.util
from pathlib import Path

# O Jupyter do VS Code anteriormente encontrava Java 8; fixar o Temurin 17 no kernel.
if os.name == 'nt':
    jdks = sorted(glob.glob(r'C:\Program Files\Eclipse Adoptium\jdk-17*'), reverse=True)
    if jdks:
        os.environ['JAVA_HOME'] = jdks[0]
        os.environ['PATH'] = str(Path(jdks[0]) / 'bin') + os.pathsep + os.environ['PATH']
        print('JAVA_HOME no kernel:', os.environ['JAVA_HOME'])
    else:
        print('ATENÇÃO: Java 17 não localizado automaticamente. Confira java -version no kernel.')

java_info = subprocess.run(['java', '-version'], capture_output=True, text=True, timeout=20)
print('Java utilizado:', (java_info.stderr or java_info.stdout).splitlines()[0])

if importlib.util.find_spec('pyspark') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark>=3.5,<5'])

import pyspark
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, StringIndexerModel
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.functions import vector_to_array
import matplotlib.pyplot as plt
import numpy as np

spark = (SparkSession.builder
         .master('local[2]')
         .appName('TSE2024-Despesas-ML-Classificacao-Corrigido')
         .config('spark.sql.shuffle.partitions', '32')
         .config('spark.sql.execution.arrow.pyspark.enabled', 'false')
         .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Python:', sys.version.split()[0], '| PySpark:', pyspark.__version__)
print('Spark:', spark.version, '| Master:', spark.sparkContext.master)
spark.range(3).show()

# Diagnóstico antes do treinamento, sem instalar executáveis externos no Windows.
TESTE_ESCRITA_OK = False
pasta_teste = (Path.cwd() / '_teste_gravacao_spark_tse').resolve()
try:
    spark.range(1).write.mode('overwrite').parquet(str(pasta_teste))
    TESTE_ESCRITA_OK = (spark.read.parquet(str(pasta_teste)).count() == 1)
    print('Teste de escrita Spark:', 'OK' if TESTE_ESCRITA_OK else 'INCONCLUSIVO')
except Exception as erro_escrita:
    print('AVISO: o Spark lê/treina, mas a gravação no sistema de arquivos falhou.')
    print('Tipo:', type(erro_escrita).__name__)
    print('Trecho do erro:', str(erro_escrita)[-900:])
    if os.name == 'nt':
        print('Possível ausência de winutils.exe / hadoop.dll compatíveis. NÃO baixar binários aleatórios.')
        print('Para concluir o requisito de save(), execute o notebook no Colab/Linux')
        print('ou configure componentes Hadoop para Windows de origem confiável.')
finally:
    if pasta_teste.exists():
        try:
            shutil.rmtree(pasta_teste)
        except OSError:
            pass


## 3. Baixar e extrair a fonte original (Unidade 4)

O download ocorre **a partir do CDN oficial do TSE**, conforme o enunciado. O CSV não será publicado no GitHub. Se o ZIP já estiver disponível localmente, use a variável `ZIP_LOCAL` na próxima célula para evitar um novo download. O notebook extrai somente os arquivos de **despesas contratadas de candidaturas**, não todo o pacote de receitas, pagamentos e outras tabelas.

A coleta não faz amostragem aleatória: lê a fonte nacional. Mais adiante há um **recorte explícito** para as cinco categorias mais frequentes *no treinamento*, com contagem e percentual retidos.

**Reutilização:** se `dados_tse_2024/prestacao_de_contas_eleitorais_candidatos_2024.zip` já existir, o download de aproximadamente 1,33 GB será ignorado. Os CSVs extraídos também serão reutilizados.


In [ ]:
from urllib.request import Request, urlopen
import zipfile
import shutil
import csv
import re
import time

URL_OFICIAL = ('https://cdn.tse.jus.br/estatistica/sead/odsele/prestacao_contas/'
               'prestacao_de_contas_eleitorais_candidatos_2024.zip')
PASTA_DADOS = Path('dados_tse_2024')
PASTA_DADOS.mkdir(exist_ok=True)
ZIP_LOCAL = PASTA_DADOS / 'prestacao_de_contas_eleitorais_candidatos_2024.zip'
# Caso possua um ZIP pré-baixado, ajuste ZIP_LOCAL para o caminho do arquivo.

if not ZIP_LOCAL.exists() or not zipfile.is_zipfile(ZIP_LOCAL):
    # Gravação gradual: evita armazenar 1+ GB na memória do Python.
    temporario = ZIP_LOCAL.with_suffix('.zip.part')
    for tentativa in range(1, 4):
        try:
            req = Request(URL_OFICIAL, headers={'User-Agent': 'Mozilla/5.0 (SparkML; pesquisa acadêmica)'})
            with urlopen(req, timeout=180) as resposta, temporario.open('wb') as destino:
                shutil.copyfileobj(resposta, destino, length=8 * 1024 * 1024)
            if not zipfile.is_zipfile(temporario):
                raise ValueError('O arquivo recebido não é um ZIP válido.')
            temporario.replace(ZIP_LOCAL)
            break
        except Exception as erro:
            if temporario.exists():
                temporario.unlink()
            print(f'Tentativa {tentativa}/3 falhou: {erro}')
            if tentativa == 3:
                raise RuntimeError('Não foi possível baixar o ZIP oficial. '
                                   'Verifique a conexão ou informe ZIP_LOCAL.') from erro
            time.sleep(5 * tentativa)

with zipfile.ZipFile(ZIP_LOCAL) as pacote:
    membros = [m for m in pacote.namelist()
               if re.search(r'(^|/)despesas_contratadas_candidatos_2024_[A-Za-z]+\.csv$',m)]
    nacional = [m for m in membros if Path(m).name.upper().endswith('_BRASIL.CSV')]
    selecionados = nacional if nacional else membros
    if not selecionados:
        raise FileNotFoundError('O ZIP não contém arquivo de despesas contratadas de candidatos de 2024. '
                                'Verifique o conteúdo e a estrutura da versão disponibilizada pelo TSE.')
    arquivos_csv = []
    for nome_membro in selecionados:
        destino = PASTA_DADOS / Path(nome_membro).name
        if not destino.exists() or destino.stat().st_size == 0:
            print('Extraindo:', Path(nome_membro).name)
            with pacote.open(nome_membro) as origem, destino.open('wb') as saida:
                shutil.copyfileobj(origem, saida, length=8 * 1024 * 1024)
        arquivos_csv.append(str(destino.resolve()))

print('Fonte:', URL_OFICIAL)
print('ZIP em disco (GB):', round(ZIP_LOCAL.stat().st_size / 1e9, 3))
print('Arquivos selecionados:', len(arquivos_csv))
print('Primeiros arquivos:', [Path(p).name for p in arquivos_csv[:5]])

## 4. Ingestão com `StructType` e inspeção inicial (Unidade 4)

O enunciado exige **schema manual ou justificativa para inferência**. Aqui **não usamos `inferSchema`**: construímos um `StructType` com as colunas declaradas no **cabeçalho real** e `StringType` como tipo inicial. Isso preserva os valores monetários no padrão brasileiro, identificadores e códigos sem arredondamento; as conversões numéricas são feitas de modo explícito no Spark.

Essa estratégia também corrige o problema observado anteriormente com colunas não existentes: **nenhum campo opcional é exigido sem conferir o cabeçalho**.

In [ ]:
def ler_cabecalho(arquivo):
    with open(arquivo, encoding='latin1', newline='') as f:
        return [c.strip().lstrip('\ufeff') for c in next(csv.reader(f, delimiter=';', quotechar='"'))]

cabecalho = ler_cabecalho(arquivos_csv[0])
for caminho in arquivos_csv[1:]:
    if ler_cabecalho(caminho) != cabecalho:
        raise ValueError('Cabeçalhos divergentes entre os CSVs de UFs. Não é seguro combinar por posição.')

schema_manual = T.StructType([T.StructField(nome, T.StringType(), True) for nome in cabecalho])
bruto = (spark.read.format('csv')
         .option('header', 'true')
         .option('sep', ';')
         .option('encoding', 'ISO-8859-1')
         .option('quote', '"')
         .option('escape', '"')
         .option('mode', 'FAILFAST')
         .option('enforceSchema', 'false')
         .schema(schema_manual)
         .load(arquivos_csv))

obrigatorias = ['DS_ORIGEM_DESPESA', 'VR_DESPESA_CONTRATADA', 'SQ_CANDIDATO', 'SG_UF']
faltantes = [c for c in obrigatorias if c not in bruto.columns]
if faltantes:
    raise ValueError('Faltam campos indispensáveis no CSV do TSE: '+str(faltantes)+
                     '\nColunas encontradas: '+str(bruto.columns))

print('Quantidade de colunas:',len(bruto.columns))
print('Nome das colunas:',bruto.columns)
bruto.printSchema()
bruto.show(5,truncate=34)

### 4.1. Dimensão real da fonte e partições (Unidades 1, 3 e 4)

- `count()` conta o número exato de linhas da fonte antes dos filtros.
- `rdd.getNumPartitions()` mostra como o Spark organizou a leitura.
- `explain()` mostra o plano de execução; DataFrames avaliam transformações de modo *lazy* até ações como `count()` ou `show()`.

In [ ]:
N_ORIGINAL = bruto.count()
print('REGISTROS originais do CSV de despesas contratadas:',f'{N_ORIGINAL:,}'.replace(',','.'))
print('COLUNAS originais:',len(bruto.columns))
print('PARTIÇÕES de leitura:',bruto.rdd.getNumPartitions())
assert N_ORIGINAL >= 200_000, 'O arquivo não tem 200 mil registros; confira se baixou todos os dados.'
bruto.select('SG_UF','DS_ORIGEM_DESPESA').groupBy('SG_UF').count().orderBy(F.desc('count')).show(30,False)

## 5. Exploração estatística, categorias e nulos (Unidade 4)

Toda a exploração do conjunto completo ocorre com funções do **Spark** (`groupBy`, `count`, `agg`, `when`, `sum`). Apenas **tabelas agregadas pequenas** são coletadas para gráficos.

As categorias oficiais da coluna `DS_ORIGEM_DESPESA` serão **mantidas literalmente**, sem agrupamentos arbitrários. Não incluiremos informações de identidade de candidatos nem identificadores de fornecedores como atributos.

O nome **`TP_PRESTACAO_CONTAS`** (e não `DS_TIPO_PRESTACAO_CONTAS`) foi conferido no esquema efetivamente carregado pelo aluno. A distribuição entre `Final` e outros tipos aparecerá na próxima etapa.


In [ ]:
bruto.groupBy('DS_ORIGEM_DESPESA').count().orderBy(F.desc('count')).show(20,False)
bruto.select('VR_DESPESA_CONTRATADA').show(5,truncate=False)
colunas_auditoria = [c for c in [
    'DS_ORIGEM_DESPESA','VR_DESPESA_CONTRATADA','SG_UF','DS_CARGO',
    'SG_PARTIDO','DS_TIPO_FORNECEDOR','DT_DESPESA','TP_PRESTACAO_CONTAS'
] if c in bruto.columns]
print('Colunas auditadas:', colunas_auditoria)
condicoes_nulos = [F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == '') |
                              F.upper(F.trim(F.col(c))).isin('#NULO','#NE','#OCULTO'),1)
                          .otherwise(0)).alias(c) for c in colunas_auditoria]
bruto.select(*condicoes_nulos).show(truncate=False)

## 6. Limpeza e auditoria de versões (Unidade 4)

O CSV contém prestações de contas com rótulos de tipo. **Somente registros `Final` entram no estudo**, evitando misturar versões parciais com versões finais. Depois:

1. Normalizar categoria e valor contratados com funções Spark.
2. Remover nulos e valores inválidos usando filtros Spark.
3. Remover somente linhas **inteiramente idênticas**.
4. Para `SQ_CANDIDATO + SQ_DESPESA`, calcular uma **assinatura dos atributos essenciais** (categoria, valor, data, UF, cargo, fornecedor quando disponível).
5. **Excluir da modelagem chaves com assinaturas divergentes**, pois não há documentação suficiente no CSV para decidir automaticamente qual lançamento é o autoritativo.
6. Consolidar as chaves repetidas com assinatura igual. A escolha da linha material é irrelevante para os atributos usados no modelo, que são iguais.

Contagens de exclusões, duplicatas e conflitos são exibidas. A seleção exige o identificador `SQ_DESPESA` e falha explicitamente se a coluna não existir; não há deduplicação silenciosa por chave inadequada.

**Limitação:** `Final` não prova que a prestação foi aprovada nem que o lançamento esteja livre de retificações posteriores. A auditoria identifica divergências sobre os campos disponíveis, não substitui conciliação documental.


In [ ]:
# Seleção da prestação: o CSV de 2024 possui TP_PRESTACAO_CONTAS (verificado no arquivo do aluno).
col_tipo = 'TP_PRESTACAO_CONTAS'
if col_tipo not in bruto.columns:
    raise ValueError('A coluna TP_PRESTACAO_CONTAS não existe. Confira o leiaute antes de continuar.')
if 'SQ_DESPESA' not in bruto.columns:
    raise ValueError('A coluna SQ_DESPESA não existe. Não é seguro deduplicar por outra chave sem auditoria.')

print('Distribuição original de TP_PRESTACAO_CONTAS:')
bruto.groupBy(col_tipo).count().orderBy(F.desc('count')).show(20, truncate=False)
tipo_normalizado = F.upper(F.trim(F.col(col_tipo)))
# Retém 'Final' e variantes que contenham essa palavra; exclui 'Parcial'.
trab = bruto.filter(tipo_normalizado.contains('FINAL') & ~tipo_normalizado.contains('PARCIAL'))
N_FINAL = trab.count()
print('Registros antes da limpeza, somente prestações Final:', N_FINAL)
assert N_FINAL > 0, 'Nenhuma prestação Final encontrada. Verifique TP_PRESTACAO_CONTAS.'

categoria_raw = F.upper(F.trim(F.col('DS_ORIGEM_DESPESA')))
valor_raw = F.trim(F.col('VR_DESPESA_CONTRATADA'))
# O TSE informa valores como texto com vírgula decimal. Exemplo: 1.234,56 -> 1234.56.
valor_normalizado = F.regexp_replace(F.regexp_replace(valor_raw, r'\.', ''), ',', '.')
trab = (trab
    .withColumn('categoria', categoria_raw)
    .withColumn('valor_reais', valor_normalizado.cast('double'))
    .filter(F.col('categoria').isNotNull() & (F.length('categoria') > 2))
    .filter(~F.col('categoria').isin('#NULO', '#NE', '#OCULTO', 'NAO INFORMADO', 'NÃO INFORMADO'))
    .filter(F.col('SQ_CANDIDATO').isNotNull() & (F.trim('SQ_CANDIDATO') != ''))
    .filter(~F.upper(F.trim('SQ_CANDIDATO')).isin('#NULO', '#NE', '#OCULTO'))
    .filter(F.col('SQ_DESPESA').isNotNull() & (F.trim('SQ_DESPESA') != ''))
    .filter(~F.upper(F.trim('SQ_DESPESA')).isin('#NULO', '#NE', '#OCULTO'))
    .filter(F.col('SG_UF').isNotNull() & (F.trim('SG_UF') != ''))
    .filter(F.col('valor_reais').isNotNull() & ~F.isnan('valor_reais'))
    .filter((F.col('valor_reais') >= 0) & (F.col('valor_reais') < 1e12)))

N_LIMPEZA_INICIAL = trab.count()
print('Registros após filtros de qualidade:', N_LIMPEZA_INICIAL)

# 1) Duplicatas estritamente idênticas (todas as colunas).
trab = trab.dropDuplicates()
N_SEM_EXATAS = trab.count()
N_DUPLICATAS_EXATAS = N_LIMPEZA_INICIAL - N_SEM_EXATAS
print('Duplicatas totalmente idênticas removidas:', N_DUPLICATAS_EXATAS)

# 2) Auditoria de possíveis versões de uma mesma despesa.
chave_duplicacao = ['SQ_CANDIDATO', 'SQ_DESPESA']
campos_assinatura = [c for c in [
    'categoria', 'VR_DESPESA_CONTRATADA', 'DT_DESPESA', 'SG_UF',
    'DS_CARGO', 'SG_PARTIDO', 'DS_TIPO_FORNECEDOR',
    'NR_CPF_CNPJ_FORNECEDOR', 'NM_FORNECEDOR'
] if c in trab.columns]

assinatura = F.sha2(F.concat_ws('||', *[
    F.concat(F.lit(c + '='), F.coalesce(F.trim(F.col(c).cast('string')), F.lit('<NULL>')))
    for c in campos_assinatura
]), 256)
com_assinatura = trab.withColumn('_assinatura_essencial', assinatura)
grupos = (com_assinatura.groupBy(*chave_duplicacao)
          .agg(F.count('*').alias('_linhas_chave'),
               F.countDistinct('_assinatura_essencial').alias('_tipos_essenciais')))
conflitos = grupos.filter(F.col('_tipos_essenciais') > 1).select(*chave_duplicacao)
N_CHAVES_CONFLITANTES = conflitos.count()
N_LINHAS_CONFLITANTES = (com_assinatura.join(conflitos, chave_duplicacao, 'inner').count())
print('Chaves com atributos essenciais divergentes:', N_CHAVES_CONFLITANTES)
print('Linhas dessas chaves (não utilizadas no modelo):', N_LINHAS_CONFLITANTES)

# Por prudência, não escolhemos arbitrariamente entre fatos divergentes.
sem_conflitos = com_assinatura.join(conflitos, chave_duplicacao, 'left_anti')
N_ANTES_CONSOLIDACAO = sem_conflitos.count()
trab = (sem_conflitos.dropDuplicates(chave_duplicacao)
        .drop('_assinatura_essencial'))
n_pos_dedupe = trab.count()
print('Repetições com atributos equivalentes consolidadas:',
      N_ANTES_CONSOLIDACAO - n_pos_dedupe)
print('Registros finais após limpeza e auditoria:', n_pos_dedupe)
print('Chave usada:', chave_duplicacao)
print('Colunas comparadas na assinatura:', campos_assinatura)
assert n_pos_dedupe >= 200_000, (
    'Menos de 200 mil após a auditoria. Reveja o recorte e as chaves; não ignore conflitos.')


## 7. Atributos numéricos e categóricos (Unidades 4 e 5)

| Variável | Origem | Tratamento |
|---|---|---|
| `valor_log` | Valor da despesa contratada | `log1p(valor_reais)` para reduzir assimetria. |
| `mes_despesa` | Data da despesa, se houver | Extração do mês. `0` indica data ausente/indisponível. |
| `SG_UF` | UF da candidatura | `StringIndexer` e `OneHotEncoder`. |
| `DS_CARGO` | Cargo (se existente) | Codificação categórica. |
| `SG_PARTIDO` | Partido (se existente) | Codificação categórica. Não é uma avaliação do partido. |
| `DS_TIPO_FORNECEDOR` | Natureza cadastral do fornecedor | Codificação categórica. |

As colunas opcionais ausentes recebem `DESCONHECIDO` e isso é **informado**, em vez de gerar `ValueError` como no notebook anterior. A categoria de origem é apenas o **rótulo**, nunca uma variável de entrada. Nenhum nome, CPF, CNPJ, identificador de despesa/candidato ou descrição textual que possa copiar o rótulo entra no modelo.

**Observação:** valores monetários são transformados com `log1p` para reduzir assimetria. Em caso de colunas opcionais ausentes, os valores `DESCONHECIDO` são explicitados em vez de inferir atributos inexistentes.


In [ ]:
CATEGORICAS = ['SG_UF','DS_CARGO','SG_PARTIDO','DS_TIPO_FORNECEDOR']
for c in CATEGORICAS:
    if c not in trab.columns:
        print('Atributo opcional ausente, preenchido como DESCONHECIDO:',c)
        trab = trab.withColumn(c,F.lit('DESCONHECIDO'))
    trab = trab.withColumn(c,F.when(F.col(c).isNull() | (F.trim(F.col(c)) == '') |
                                      F.upper(F.trim(F.col(c))).isin('#NULO','#NE','#OCULTO'),
                                      'DESCONHECIDO').otherwise(F.upper(F.trim(F.col(c)))))

DATA_ENCONTRADA = next((x for x in ['DT_DESPESA','DT_DESPESA_CONTRATADA'] if x in trab.columns),None)
if DATA_ENCONTRADA:
    print('Data usada:',DATA_ENCONTRADA)
    texto_data = F.trim(F.col(DATA_ENCONTRADA))
    data_convertida = F.coalesce(F.to_date(texto_data,'dd/MM/yyyy'),
                                 F.to_date(texto_data,'yyyy-MM-dd'))
    trab = trab.withColumn('mes_despesa', F.coalesce(F.month(data_convertida),F.lit(0)).cast('double'))
else:
    print('Campo de data não identificado: mes_despesa=0 (variável constante).')
    trab = trab.withColumn('mes_despesa',F.lit(0.0))

trab = trab.withColumn('valor_log',F.log1p(F.col('valor_reais')))
NUMERICAS = ['valor_log','mes_despesa']
trab.select('valor_reais','valor_log','mes_despesa').describe().show(truncate=False)
trab.groupBy('categoria').agg(F.count('*').alias('registros'),
                               F.round(F.avg('valor_reais'),2).alias('valor_medio_reais')) \
    .orderBy(F.desc('registros')).show(15,truncate=False)

## 7.1. Amostragem opcional e reproduzível para executar no Colab

O arquivo nacional completo contém milhões de linhas; a quantidade real após a filtragem Final será exibida. O enunciado **permite amostragem** desde que informe o tamanho original e o procedimento. Por padrão, quando há mais de **1,2 milhão** de registros após limpeza, selecionamos aproximadamente **700 mil** usando um hash determinístico do **identificador da candidatura** — todas as despesas de uma candidatura entram ou saem juntas. Essa é uma **amostra por conglomerados**, não aleatória simples de linhas; pode distorcer levemente a distribuição se as candidaturas tiverem quantidades muito diferentes de despesas.

Isso conserva a reprodutibilidade, evita contaminação entre grupos e reduz o custo computacional do CrossValidator. O número real da amostra e o percentual retido serão impressos. Para trabalhar com **100%** dos dados após limpeza, defina `ATIVAR_AMOSTRAGEM_AUTOMATICA = False`, com consciência do custo de memória e processamento.


In [ ]:
ATIVAR_AMOSTRAGEM_AUTOMATICA = True
LIMIAR_AMOSTRAGEM = 1_200_000
ALVO_LINHAS_AMOSTRA = 700_000

N_ANTES_AMOSTRA = n_pos_dedupe
if ATIVAR_AMOSTRAGEM_AUTOMATICA and N_ANTES_AMOSTRA > LIMIAR_AMOSTRAGEM:
    fracao_planejada = ALVO_LINHAS_AMOSTRA / N_ANTES_AMOSTRA
    limite_hash = max(1,int(fracao_planejada * 100_000))
    # Amostragem sistemática por hash determinístico de grupos/candidaturas.
    condicao = F.pmod(F.xxhash64(F.col('SQ_CANDIDATO'),F.lit('amostragem_tse_2024')),
                      F.lit(100_000)) < F.lit(limite_hash)
    trab = trab.filter(condicao)
    AMOSTRAGEM_DESCRICAO = (f'Por conglomerados de SQ_CANDIDATO; hash de 0 a {limite_hash-1} '
                            'no intervalo 0..99999')
else:
    AMOSTRAGEM_DESCRICAO = 'Censo dos registros limpos (sem amostragem)'

N_APOS_AMOSTRA = trab.count()
print('Total no CSV nacional:',N_ORIGINAL)
print('Total após limpeza/duplicatas:',N_ANTES_AMOSTRA)
print('Total na amostra para modelagem:',N_APOS_AMOSTRA)
print('Percentual conservado dos limpos:',round(100*N_APOS_AMOSTRA/N_ANTES_AMOSTRA,2),'%')
print('Procedimento:',AMOSTRAGEM_DESCRICAO)
assert N_APOS_AMOSTRA >= 200_000, (
    'A amostragem resultou em menos de 200 mil linhas. Desative a amostragem '
    'ou eleve ALVO_LINHAS_AMOSTRA e execute de novo.')


## 8. Divisão por candidatura para impedir vazamento (Unidade 5)

O mesmo `SQ_CANDIDATO` pode ter muitas despesas. Uma divisão linha a linha permitiria que despesas da **mesma campanha** aparecessem tanto no treino quanto no teste. Para uma avaliação mais rigorosa, separamos **candidaturas inteiras** por hash determinístico em aproximadamente **70% treino, 15% validação e 15% teste**. As porcentagens de registros podem diferir devido ao tamanho de cada grupo.

**Seleção das classes:** após a amostragem por candidatura descrita na seção anterior (quando ativada), utilizamos as **cinco categorias mais frequentes apenas no treino**. Aplicamos essas categorias fixas às três partes. Esse recorte é uma **seleção por frequência, não amostragem aleatória**, e informamos o volume descartado. O conjunto de teste permanece intacto para a avaliação final do modelo escolhido.

A seleção das cinco classes usa **apenas a parte de treinamento**. O percentual de cobertura deve ter como denominador a amostra já produzida, e não o conjunto completo antes da amostragem.


In [ ]:
N_CLASSES = 5
SEMENTE = 42

hash_grupo = F.pmod(F.xxhash64(F.col('SQ_CANDIDATO')),F.lit(100))
com_particao = trab.withColumn('_bucket',hash_grupo)
selecao_treino = com_particao.filter(F.col('_bucket') < 70)

top_categorias = [r['categoria'] for r in
    selecao_treino.groupBy('categoria').count().orderBy(F.desc('count'),F.asc('categoria'))
    .limit(N_CLASSES).collect()]
if len(top_categorias) != N_CLASSES:
    raise ValueError('Número insuficiente de categorias: '+str(top_categorias))

modelo_base = com_particao.filter(F.col('categoria').isin(top_categorias))
N_RECORTADO = modelo_base.count()
print('Categorias selecionadas com base SOMENTE no treino:')
for idx, categoria in enumerate(top_categorias,1):
    print(f'{idx}. {categoria}')
print('Registros após limpeza (todas as categorias):',n_pos_dedupe)
print('Registros nas classes selecionadas:',N_RECORTADO)
print('Cobertura das 5 classes NA AMOSTRA:',round(100*N_RECORTADO/N_APOS_AMOSTRA,2),'%')
assert N_RECORTADO >= 200_000, ('O recorte final tem menos de 200 mil registros. '
                                'Aumente N_CLASSES, sem amostragem aleatória.')

treino = modelo_base.filter(F.col('_bucket') < 70)
validacao = modelo_base.filter((F.col('_bucket') >= 70) & (F.col('_bucket') < 85))
teste = modelo_base.filter(F.col('_bucket') >= 85)
print('Treino:',treino.count(),'| Validação:',validacao.count(),'| Teste:',teste.count())
for nome, dados in [('Treino',treino),('Validação',validacao),('Teste',teste)]:
    print('Classes presentes em',nome,':',dados.select('categoria').distinct().count())

### 8.1. Distribuição das classes e desbalanceamento

Classe majoritária elevada pode inflar a **acurácia** de um classificador ingênuo. Vamos apresentar contagens de todas as classes, além das métricas macro-F1 e F1 ponderado. Os cálculos abaixo são agregados no Spark.

In [ ]:
contagens = (modelo_base.groupBy('categoria').count()
             .orderBy(F.desc('count')).collect())
nomes_plot = [r['categoria'] for r in contagens]
quant_plot = [r['count'] for r in contagens]
plt.figure(figsize=(11,4))
plt.barh(range(len(nomes_plot)),quant_plot)
plt.yticks(range(len(nomes_plot)),[x[:48] for x in nomes_plot])
plt.gca().invert_yaxis()
plt.xlabel('Número de registros')
plt.title('Categorias de despesa selecionadas (TSE, 2024)')
plt.tight_layout()
plt.show()
for r in contagens:
    print(f"{r['categoria']}: {r['count']:,} ({100*r['count']/N_RECORTADO:.1f}%)")

## 9. Rótulo codificado sem consultar o teste (Unidade 5)

`StringIndexer` ajusta a correspondência entre categorias textuais e índices inteiros **somente nos dados de treino**. O modelo da codificação do rótulo será salvo separadamente do classificador. Assim, a pipeline de predição não precisará da coluna `categoria` quando receber novos registros.

In [ ]:
indexador_label = StringIndexer(inputCol='categoria',outputCol='label',
                               handleInvalid='error',stringOrderType='frequencyDesc')
indexador_label_model = indexador_label.fit(treino)
nomes_classes = list(indexador_label_model.labels)
treino = indexador_label_model.transform(treino)
validacao = indexador_label_model.transform(validacao)
teste = indexador_label_model.transform(teste)
print('Correspondência índice -> classe:')
for i,nome in enumerate(nomes_classes):
    print(i,'=>',nome)

## 10. Pipeline de atributos e algoritmos (Unidade 5)

Aplicamos exatamente os componentes apresentados nas aulas:

1. `StringIndexer` para cada atributo categórico (com tratamento de valores não vistos).
2. `OneHotEncoder` para evitar impor uma ordenação artificial entre UFs, cargos etc.
3. `VectorAssembler` para combinar categóricas e numéricas.
4. `StandardScaler` para escalar as colunas do vetor; com `withMean=False` preserva a representação esparsa.
5. Classificador supervisionado: **Regressão Logística multinomial** ou **Random Forest**.

Todos os transformadores que aprendem parâmetros (`StringIndexer`, `OneHotEncoder`, `StandardScaler`) são ajustados **apenas dentro do `fit()` de treino**, evitando vazamento de pré-processamento.

In [ ]:
def criar_pipeline(classificador):
    colunas_indices = [f'{c}_idx' for c in CATEGORICAS]
    colunas_ohe = [f'{c}_ohe' for c in CATEGORICAS]
    etapas = [StringIndexer(inputCol=c,outputCol=idx,handleInvalid='keep')
              for c,idx in zip(CATEGORICAS,colunas_indices)]
    etapas.append(OneHotEncoder(inputCols=colunas_indices,outputCols=colunas_ohe,
                                handleInvalid='keep'))
    etapas.append(VectorAssembler(inputCols=colunas_ohe+NUMERICAS,
                                  outputCol='features_brutas',handleInvalid='keep'))
    etapas.append(StandardScaler(inputCol='features_brutas',outputCol='features',
                                 withStd=True,withMean=False))
    etapas.append(classificador)
    return Pipeline(stages=etapas)

lr = LogisticRegression(featuresCol='features',labelCol='label',
                        family='multinomial',maxIter=45,regParam=0.05)
rf = RandomForestClassifier(featuresCol='features',labelCol='label',
                            numTrees=35,maxDepth=8,maxBins=64,seed=SEMENTE)
pipe_lr = criar_pipeline(lr)
pipe_rf = criar_pipeline(rf)
print('Etapas do Pipeline:',[type(x).__name__ for x in pipe_lr.getStages()])

## 11. Referência ingênua e métricas (Unidade 5)

Para impedir conclusões equivocadas, usamos uma **baseline**: sempre prever a categoria majoritária **no treino**. Avaliamos:

- **Acurácia:** percentual de classificações corretas (pode ser enganador com classes desequilibradas).
- **F1 ponderado:** pondera o F1 pelo suporte de cada classe.
- **Macro-F1:** dá o mesmo peso a cada uma das cinco classes, independentemente de frequência.
- **AUC-ROC macro One-vs-Rest:** em cada classe, considera-a positiva contra todas as demais; usa `probability[k]` como score. É a generalização apropriada do AUC para nosso problema **multiclasse**, não a AUC binária aplicada diretamente a cinco rótulos.
- **Matriz de confusão** e precisão/recall/F1 por categoria.

As métricas de validação servem para **escolher o modelo**; a parte de teste só será avaliada no final.

In [ ]:
def avaliar_multiclasse(df_com_pred):
    valores = {}
    for nome,metrica in [('acuracia','accuracy'),('f1_ponderado','f1'),
                          ('precisao_ponderada','weightedPrecision'),
                          ('recall_ponderado','weightedRecall')]:
        ev = MulticlassClassificationEvaluator(
            labelCol='label',predictionCol='prediction',metricName=metrica)
        valores[nome] = float(ev.evaluate(df_com_pred))
    # "f1" do Spark é F1 ponderado; macro-F1 é calculado corretamente abaixo.
    pares = {(int(r['label']),int(r['prediction'])):int(r['count'])
             for r in df_com_pred.groupBy('label','prediction').count().collect()}
    n = len(nomes_classes)
    por_classe = []
    for k in range(n):
        tp = pares.get((k,k),0)
        fp = sum(pares.get((j,k),0) for j in range(n) if j != k)
        fn = sum(pares.get((k,j),0) for j in range(n) if j != k)
        precision = tp/(tp+fp) if tp+fp else 0.0
        recall = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
        por_classe.append({'indice':k,'classe':nomes_classes[k],
                           'suporte':tp+fn,'precisao':precision,
                           'recall':recall,'f1':f1})
    valores['f1_macro'] = float(np.mean([x['f1'] for x in por_classe]))
    return valores,por_classe,pares

index_majoritaria = (treino.groupBy('label').count().orderBy(F.desc('count')).first()['label'])
baseline_val = validacao.withColumn('prediction',F.lit(float(index_majoritaria)))
metricas_baseline_val,_,_ = avaliar_multiclasse(baseline_val)
print('Baseline em validação:',metricas_baseline_val)

## 12. Ajuste de hiperparâmetros com `ParamGridBuilder` e `CrossValidator` (Unidade 5)

O exemplo do professor usa `ParamGridBuilder` e `CrossValidator`. Testamos **duas regularizações** para regressão logística, com **3 folds**. O fold é determinado por hash de `SQ_CANDIDATO`, mantendo despesas da **mesma candidatura no mesmo fold**.

Na validação externa, comparamos essa regressão logística com Random Forest. A escolha usa **macro-F1**, métrica que protege contra um modelo que acerta apenas a classe mais comum. Os hiperparâmetros são deliberadamente limitados para execução no Colab; aumentar a grade eleva o custo computacional.

O `CrossValidator` utiliza F1 **ponderado** nos folds; a comparação entre algoritmos utiliza **macro-F1** na validação independente. Isso é intencional e está identificado no código.


In [ ]:
# foldCol é alimentada por um valor inteiro [0, 1, 2] para cada candidatura.
fold = F.pmod(F.xxhash64(F.col('SQ_CANDIDATO'),F.lit('fold_cv_2024')),F.lit(3)).cast('int')
treino_cv = treino.withColumn('_fold_cv',fold).cache()
print('Treino por fold:')
treino_cv.groupBy('_fold_cv').count().orderBy('_fold_cv').show()

parametros = (ParamGridBuilder()
              .addGrid(lr.regParam,[0.03,0.25])
              .build())
validador = MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',
                                               metricName='f1')
cv = CrossValidator(estimator=pipe_lr,estimatorParamMaps=parametros,
                    evaluator=validador,numFolds=3,foldCol='_fold_cv',
                    parallelism=2,seed=SEMENTE)
print('Ajustando Regressão Logística (3 folds × 2 regularizações)...')
cv_lr = cv.fit(treino_cv)
print('Métricas médias dos folds (F1 ponderado):',cv_lr.avgMetrics)
print('Regularização escolhida:',cv_lr.bestModel.stages[-1].getRegParam())

print('Treinando Random Forest...')
modelo_rf = pipe_rf.fit(treino_cv)

## 13. Comparação de modelos no conjunto de validação (Unidade 5)

A tabela contém apenas valores **efetivamente calculados**. O modelo com maior macro-F1 na validação é escolhido; se houver empate, a regressão logística (mais simples) fica como padrão. **O teste não participa da escolha.**

In [ ]:
pred_lr_val = cv_lr.bestModel.transform(validacao).cache()
pred_rf_val = modelo_rf.transform(validacao).cache()
metricas_lr_val,_,_ = avaliar_multiclasse(pred_lr_val)
metricas_rf_val,_,_ = avaliar_multiclasse(pred_rf_val)
comparacao = [('Baseline majoritária',metricas_baseline_val),
              ('Regressão logística',metricas_lr_val),
              ('Random Forest',metricas_rf_val)]
print('MODELO'.ljust(27),'Acurácia    Macro-F1    F1 ponderado')
for nome,m in comparacao:
    print(f"{nome:27s} {m['acuracia']:.4f}      {m['f1_macro']:.4f}      {m['f1_ponderado']:.4f}")

ESCOLHIDO = ('Regressão logística' if metricas_lr_val['f1_macro'] >= metricas_rf_val['f1_macro']
             else 'Random Forest')
print('Modelo selecionado sem consultar teste:',ESCOLHIDO)

## 14. Ajuste final e teste independente (Unidade 5)

Agora reunimos **treino + validação** e ajustamos apenas o algoritmo escolhido, reutilizando os hiperparâmetros já definidos na fase de treino/validação. O conjunto de teste continuou isolado até aqui.

O rótulo permanece codificado com o `StringIndexerModel` ajustado anteriormente. Guardamos as previsões finais para métricas, matriz de confusão e interpretação.

In [ ]:
treino_final = treino.unionByName(validacao).cache()
if ESCOLHIDO == 'Regressão logística':
    reg_final = cv_lr.bestModel.stages[-1].getRegParam()
    classificador_final = LogisticRegression(featuresCol='features',labelCol='label',
                                            family='multinomial',maxIter=45,
                                            regParam=reg_final)
else:
    classificador_final = RandomForestClassifier(featuresCol='features',labelCol='label',
                                                 numTrees=35,maxDepth=8,maxBins=64,
                                                 seed=SEMENTE)

pipeline_final = criar_pipeline(classificador_final)
modelo_final = pipeline_final.fit(treino_final)
pred_teste = modelo_final.transform(teste).cache()
metricas_teste,por_classe,matriz = avaliar_multiclasse(pred_teste)
print('TESTE INDEPENDENTE —',ESCOLHIDO)
for chave in ['acuracia','f1_ponderado','f1_macro','precisao_ponderada','recall_ponderado']:
    print(f'{chave:22s}: {metricas_teste[chave]:.4f}')

## 15. AUC-ROC *One-vs-Rest* multiclasses (Unidade 5 e enunciado)

`BinaryClassificationEvaluator` não deve receber diretamente rótulos de 0 a 4. Para cada categoria **k**, criamos `label_ovr = 1` quando o rótulo é **k** e calculamos o escore pela componente `probability[k]`. A **média simples** das cinco AUCs é a AUC macro. Esta etapa também reporta **AUC-PR macro**, útil quando as categorias são desbalanceadas.

In [ ]:
prob = vector_to_array(F.col('probability'))
auc_roc = []
auc_pr = []
for k,nome in enumerate(nomes_classes):
    um_contra_resto = pred_teste.select(
        (F.col('label') == F.lit(float(k))).cast('double').alias('label_ovr'),
        prob[k].cast('double').alias('score_ovr'))
    ev_roc = BinaryClassificationEvaluator(labelCol='label_ovr',rawPredictionCol='score_ovr',
                                           metricName='areaUnderROC')
    ev_pr = BinaryClassificationEvaluator(labelCol='label_ovr',rawPredictionCol='score_ovr',
                                          metricName='areaUnderPR')
    roc = float(ev_roc.evaluate(um_contra_resto))
    pr = float(ev_pr.evaluate(um_contra_resto))
    auc_roc.append(roc)
    auc_pr.append(pr)
    print(f'Classe {k}: AUC-ROC={roc:.4f} | AUC-PR={pr:.4f} | {nome}')
AUC_ROC_MACRO = float(np.mean(auc_roc))
AUC_PR_MACRO = float(np.mean(auc_pr))
print(f'AUC-ROC macro OvR = {AUC_ROC_MACRO:.4f}')
print(f'AUC-PR  macro OvR = {AUC_PR_MACRO:.4f}')

## 16. Matriz de confusão e avaliação por categoria (Unidade 5)

As células identificam os erros de classificação entre as categorias oficiais, permitindo entender **quais tipos de despesa são confundidos**. Usamos apenas uma matriz agregada de **5 × 5**, não o conjunto bruto inteiro no Python local.

In [ ]:
print('CLASSE'.ljust(55),'SUPORTE','PRECISÃO','RECALL','F1')
for linha in por_classe:
    print(f"{linha['classe'][:55]:55s} {linha['suporte']:7d} "
          f"{linha['precisao']:.3f}    {linha['recall']:.3f}  {linha['f1']:.3f}")

qtd = len(nomes_classes)
cm = np.zeros((qtd,qtd),dtype=int)
for (verdadeiro,predito),freq in matriz.items():
    cm[verdadeiro,predito]=freq
fig,ax = plt.subplots(figsize=(9,7))
figura = ax.imshow(cm,cmap='Blues')
ax.set_xticks(range(qtd))
ax.set_yticks(range(qtd))
ax.set_xticklabels([f'C{i}' for i in range(qtd)])
ax.set_yticklabels([f'C{i}' for i in range(qtd)])
ax.set_xlabel('Classe prevista')
ax.set_ylabel('Classe real')
ax.set_title('Matriz de confusão — teste independente')
for i in range(qtd):
    for j in range(qtd):
        ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=10,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(figura,ax=ax,label='Quantidade de despesas')
plt.tight_layout()
plt.show()
print('Legenda das classes (C0...):')
for i,nome in enumerate(nomes_classes):
    print('C'+str(i)+':',nome)

## 17. Comparação com referência simples no mesmo teste

A **baseline do teste** é calculada agora, depois que a seleção do modelo já foi concluída, para comparar exatamente as mesmas observações. O objetivo é verificar se os modelos realmente superam a estratégia de classificar tudo como a categoria mais frequente.

A matriz de confusão detalha quais categorias o modelo confunde. F1 baixo em alguma classe significa que **não é adequado dizer que o sistema classifica todas as naturezas de despesa com boa qualidade**, mesmo quando a acurácia global é maior que a baseline.


In [ ]:
base_teste = teste.withColumn('prediction',F.lit(float(index_majoritaria)))
metrics_base_teste,_,_ = avaliar_multiclasse(base_teste)
print('                 Baseline          Modelo final           Diferença')
for metrica in ['acuracia','f1_ponderado','f1_macro']:
    val_b = metrics_base_teste[metrica]
    val_m = metricas_teste[metrica]
    print(f'{metrica:18s} {val_b:8.4f}          {val_m:8.4f}           {val_m-val_b:+.4f}')

grafico_nomes = ['Baseline','Regressão logística','Random Forest']
grafico_f1 = [metricas_baseline_val['f1_macro'],metricas_lr_val['f1_macro'],metricas_rf_val['f1_macro']]
plt.figure(figsize=(8,3.8))
plt.bar(grafico_nomes,grafico_f1)
plt.ylabel('Macro-F1 (validação)')
plt.title('Comparação sem usar o teste na seleção')
plt.ylim(0,1)
plt.xticks(rotation=12,ha='right')
plt.tight_layout()
plt.show()

## 18. Interpretação adicional por UF (Unidades 1, 4 e 5)

Avaliamos **taxa de erro por UF** como auditoria de generalização geográfica. Esse cálculo não atribui qualidade, culpa ou regularidade a nenhuma candidatura/partido; apenas mede o funcionamento do **classificador de categorias** entre os registros disponíveis. Para evitar taxas instáveis, exibimos somente UFs com **pelo menos 100 registros no teste**.

In [ ]:
erro_uf = (pred_teste.groupBy('SG_UF')
           .agg(F.count('*').alias('n'),
                F.avg(F.when(F.col('label')==F.col('prediction'),F.lit(0.0))
                      .otherwise(F.lit(1.0))).alias('taxa_erro'))
           .filter(F.col('n')>=100)
           .orderBy(F.desc('taxa_erro')))
erro_uf.show(27,False)

## 19. Persistência do Pipeline (Unidade 5): `save()` e recarregamento

O enunciado exige `save()` do Spark ML. Executaremos o salvamento **do Pipeline completo** e do `StringIndexerModel` de classes, seguido de `load()` e uma predição de teste.

**Importante no Windows:** o aluno confirmou que o Spark 4.2.0 treina normalmente, mas recebeu `Py4JJavaError` em `org.apache.hadoop.util.Shell.checkHadoopHomeInner` ao salvar. Isso aponta para componentes nativos de acesso ao sistema de arquivos que faltam no Windows (Hadoop `winutils.exe` / `hadoop.dll`), e **não para um defeito no modelo treinado**.

- Se o teste de gravação do início tiver funcionado, esta célula deve executar o `save()` normalmente.
- Se a gravação falhar, o notebook informará a pendência e **não afirmará que o modelo foi persistido**. A solução mais simples e segura para atender ao enunciado é executar o notebook no Google Colab/Linux ou configurar componentes Hadoop Windows compatíveis obtidos de fonte confiável.
- Não usamos `pickle`, nem arquivos JSON de coeficientes, como substitutos de `save()` exigido pelo professor.


In [ ]:
PASTA_MODELO = Path('modelos') / 'tse_2024_natureza_despesas'
PASTA_MODELO.mkdir(parents=True, exist_ok=True)
caminho_pipeline = str((PASTA_MODELO / 'pipeline').resolve())
caminho_rotulos = str((PASTA_MODELO / 'indexador_categorias').resolve())

PERSISTENCIA_OK = False
PERSISTENCIA_ERRO = None

if not TESTE_ESCRITA_OK:
    PERSISTENCIA_ERRO = (
        'Falha detectada na gravação inicial do Spark/Hadoop. '
        'No Windows, confira os componentes nativos do Hadoop. '
        'Para demonstrar save/load, execute integralmente em Colab ou Linux.'
    )
    print('PERSISTÊNCIA PENDENTE:', PERSISTENCIA_ERRO)
else:
    try:
        # Persistir com Spark ML (não substituir por pickle/JSON).
        modelo_final.write().overwrite().save(caminho_pipeline)
        indexador_label_model.write().overwrite().save(caminho_rotulos)

        carregado = PipelineModel.load(caminho_pipeline)
        indexador_carregado = StringIndexerModel.load(caminho_rotulos)
        if list(indexador_carregado.labels) != list(nomes_classes):
            raise AssertionError('Classes recarregadas diferem das classes treinadas.')

        amostra_inferencia = teste.select(*(CATEGORICAS + NUMERICAS)).limit(5)
        print('Modelo salvo e recarregado:', caminho_pipeline)
        print('Indexador salvo e recarregado:', caminho_rotulos)
        print('Rótulos recarregados:', list(indexador_carregado.labels))
        carregado.transform(amostra_inferencia) \
            .select(*CATEGORICAS, 'prediction', 'probability').show(5, truncate=24)
        PERSISTENCIA_OK = True
    except Exception as erro:
        # Não confundir treino concluído com persistência concluída.
        PERSISTENCIA_ERRO = f'{type(erro).__name__}: {str(erro)[-1000:]}'
        print('PERSISTÊNCIA PENDENTE:', PERSISTENCIA_ERRO)
        print('O treinamento pode estar correto; verifique disco, permissões e Hadoop.')
        if os.name == 'nt':
            print('Não baixe winutils.exe/hadoop.dll de repositórios desconhecidos.')

print('Status da persistência:', 'CONCLUÍDA' if PERSISTENCIA_OK else 'PENDENTE')

## 20. Conclusões geradas pelos dados efetivamente processados

Este trabalho **não presume** alta acurácia. As características disponíveis (valor e contexto) podem ser insuficientes para inferir a natureza concreta de uma contratação; a descrição de um produto/serviço foi intencionalmente omitida por potencial contaminação com o rótulo oficial. As métricas são **associação estatística**, não prova causal, e não servem para classificar a regularidade de prestações de contas.

**Limitações a discutir:**

- Despesas **contratadas** não são sinônimo de **pagas**.
- Eventual amostragem por candidatura é identificada e documentada, além do recorte pelas cinco classes mais frequentes, que exclui outras categorias.
- Dados de 2024 descrevem apenas um período eleitoral e podem ser retificados.
- Categorias não informadas e valores inválidos são excluídos; a exclusão pode não ser aleatória.
- Registros com chaves de despesa e atributos essenciais contraditórios são excluídos do experimento para evitar escolhas arbitrárias; quantidades aparecem na auditoria.
- O experimento roda no Colab, em modo local, embora use a API de Spark distribuído.
- As classes apresentam desbalanceamento e a coluna de tipo de fornecedor pode estar incompleta.

**Melhorias possíveis:** ampliar classes, verificar versões de lançamentos com chave/data documental, validar em outra eleição (2020 ou 2022), testar representação textual independente **após auditoria de vazamento**, testar outros classificadores e executar em cluster real.
- São utilizados apenas lançamentos com `TP_PRESTACAO_CONTAS` contendo **Final**; o conjunto original pode conter registros parciais que foram excluídos.
- O conjunto de **cinco classes** não representa todos os tipos possíveis de despesa. O modelo é um classificador condicionado a esse universo restrito.
- AUC-ROC multiclasses elevada não elimina recall baixo de classes minoritárias; mostrar F1 por classe é indispensável.
- A persistência somente está concluída quando a variável `PERSISTENCIA_OK` for verdadeira.


### Observação sobre a execução documentada

Na execução local mostrada pelo autor, o teste independente da regressão logística
apresentou acurácia de **0,6231**, F1 ponderado de **0,5730** e macro-F1 de
**0,4616**, com AUC-ROC OvR macro de **0,8468**. A categoria
**Serviços prestados por terceiros** teve recall de aproximadamente **0,008**,
o que impede interpretar o desempenho global como bom para todas as categorias.
Estes são **resultados observados na execução anterior**, e não resultados
produzidos novamente por este arquivo revisado; a reexecução deve recalcular
todas as métricas.


In [ ]:
print('===== RESUMO PARA O RELATÓRIO / APRESENTAÇÃO =====')
print('Fonte: TSE — Prestação de Contas Eleitorais 2024 (SPCE)')
print('Registros na origem:',N_ORIGINAL,'| Colunas na origem:',len(bruto.columns))
print('Registros com prestação Final (pré-limpeza):', N_FINAL)
print('Registros após limpeza/auditoria:', n_pos_dedupe)
print('Chaves conflitantes descartadas:', N_CHAVES_CONFLITANTES)
print('Linhas conflitantes descartadas:', N_LINHAS_CONFLITANTES)
print('Amostragem:',AMOSTRAGEM_DESCRICAO)
print('Registros após amostragem:',N_APOS_AMOSTRA)
print('Registros nas 5 classes:',N_RECORTADO,
      f'({100*N_RECORTADO/N_APOS_AMOSTRA:.1f}% da amostra)')
print('Modelo escolhido:',ESCOLHIDO)
print(f"Acurácia de teste: {metricas_teste['acuracia']:.4f}")
print(f"F1 ponderado de teste: {metricas_teste['f1_ponderado']:.4f}")
print(f"Macro-F1 de teste: {metricas_teste['f1_macro']:.4f}")
print(f'AUC-ROC macro OvR: {AUC_ROC_MACRO:.4f}')
print(f'AUC-PR  macro OvR: {AUC_PR_MACRO:.4f}')
print(f"Baseline macro-F1: {metrics_base_teste['f1_macro']:.4f}")
melhora=metricas_teste['f1_macro']-metrics_base_teste['f1_macro']
print('Diferença de macro-F1 do modelo para a baseline:',round(melhora,4))
if melhora > 0:
    print('Interpretação: o modelo superou a baseline em macro-F1 no teste.')
elif melhora < 0:
    print('Interpretação: o modelo não superou a baseline em macro-F1 no teste.')
else:
    print('Interpretação: o modelo empatou com a baseline em macro-F1 no teste.')
print('IMPORTANTE: os resultados se limitam à classificação da natureza dos lançamentos declarados.')
print('Persistência Spark ML:', 'CONCLUÍDA' if PERSISTENCIA_OK else 'PENDENTE — verificar ambiente Hadoop/Windows')


## 21. Unidade 6: batch vs Structured Streaming

A **Unidade 6** apresenta `readStream()`, *event time*, *processing time*, janelas e *watermarks*. Essas técnicas são importantes quando **novos registros chegam continuamente**, mas **não são exigidas** no enunciado deste trabalho.

Este experimento é **batch**:

`ZIP oficial (dados históricos) → spark.read.csv() → limpeza → Pipeline do Spark ML → avaliação → save()`.

**Extensão futura, não implementada:** caso uma instituição disponha de um fluxo autorizado e estruturado de lançamentos, seria possível usar Structured Streaming para leitura incremental, preservando o modelo previamente treinado. Não alegamos que o portal oficial do TSE disponibilize uma API de streaming.

## 22. Checklist de entrega / GitHub

- [ ] Executar **todas** as células no Colab ou no VS Code/Windows com Spark configurado (resultados de contagem, treinamento e avaliação devem aparecer).
- [ ] Confirmar **pelo menos 200 mil registros** após a seleção das classes.
- [ ] Verificar a matriz de confusão e os gráficos gerados.
- [ ] Confirmar que `TP_PRESTACAO_CONTAS` foi filtrado para `Final` e registrar o número de chaves excluídas por conflito.
- [ ] Confirmar que a cobertura das classes usa a amostra como denominador.
- [ ] Confirmar `PERSISTENCIA_OK = True`; se necessário, executar no Colab/Linux para concluir `save()` e `load()`.
- [ ] Deixar código, justificativas, métricas, fonte e conclusão no `.ipynb`.
- [ ] **Não enviar** ZIP, CSV, `dados_tse_2024/`, `modelos/`, caches ou checkpoints ao GitHub.
- [ ] Publicar apenas notebook, README, `requirements.txt` e `.gitignore`.
- [ ] Entregar **link do repositório GitHub** conforme o enunciado.

**Referências externas:**

1. Tribunal Superior Eleitoral. [Prestação de Contas Eleitorais — 2024](https://dadosabertos.tse.jus.br/dataset/prestacao-de-contas-eleitorais-2024).
2. Tribunal Superior Eleitoral. [Prestação de Contas — Eleições 2024](https://www.tse.jus.br/eleicoes/eleicoes-2024-content/prestacao-de-contas/prestacao-de-contas-eleicoes-2024).
3. Apache Spark. [ML Pipelines](https://spark.apache.org/docs/3.5.0/ml-pipeline.html) e [Classificação e avaliação](https://spark.apache.org/docs/3.5.0/ml-classification-regression.html).
4. Alexandre Roriz. *Processamento de Dados Massivos* — Unidades 1 a 6 (materiais fornecidos em aula).
5. Alexandre Roriz. *Construção de modelos preditivos em dados massivos usando Spark ML* (enunciado da atividade).

**Atenção:** métricas só têm validade após execução completa do notebook com o CSV nacional e conferência do leiaute efetivamente disponibilizado pelo TSE.

**Aviso sobre execução:** publicar o notebook no GitHub não equivale a concluir
a persistência do modelo. Se `PERSISTENCIA_OK=False`, o arquivo deixa a etapa
de `save()/load()` explicitamente pendente, sem mascarar a falha do ambiente.
